# Download Workshop Materials

This notebook downloads the AI Data Analyst Workshop materials from GitHub Releases.

## What This Does
1. Detects your environment (Databricks or local)
2. Downloads the workshop materials tarball from GitHub
3. Extracts to your workspace directory
4. Verifies the download

## Requirements
- Internet access to GitHub
- Write permissions to target directory

In [ ]:
# Environment Detection and Configuration
import os
import subprocess

IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

# Repository configuration
REPO = "compassanalytics/ai-data-analyst-demo"
DEFAULT_VERSION = "v1.0"

if IN_DATABRICKS:
    # Set up widgets for configuration
    try:
        dbutils.widgets.removeAll()
    except Exception:
        pass

    dbutils.widgets.text("1_version", DEFAULT_VERSION, "1. Workshop Version")

    # Auto-detect user workspace path
    user = spark.sql("SELECT current_user()").first()[0]
    default_target = f"/Workspace/Users/{user}/ai-data-analyst-workshop"
    dbutils.widgets.text("2_target_path", default_target, "2. Target Path")

    print("Running in Databricks")
    print("")
    print("Configure using the widgets above, then run the next cell.")
    print("")
    print("Widget options:")
    print("  1. Version: Workshop release version (e.g., v1.0, v1.1)")
    print(f"  2. Target Path: Where to extract materials (default: {default_target})")
else:
    print("Running locally")
    print(f"Default version: {DEFAULT_VERSION}")
    print("Materials will be downloaded to current directory.")

In [ ]:
# Download and Extract Workshop Materials
import os


def download_workshop_materials(version: str, target_path: str) -> bool:
    """Download and extract workshop materials from GitHub Releases.

    Args:
        version: Release version (e.g., "v1.0")
        target_path: Directory to extract materials to

    Returns:
        True if successful, False otherwise
    """
    release_url = f"https://github.com/{REPO}/releases/download/workshop-{version}/workshop-materials-{version}.tar.gz"

    print("Downloading workshop materials...")
    print(f"  Version: {version}")
    print(f"  URL: {release_url}")
    print(f"  Target: {target_path}")
    print("")

    try:
        # Create target directory
        os.makedirs(target_path, exist_ok=True)

        # Download and extract in one command
        cmd = f"curl -sL {release_url} | tar -xz -C {target_path} --strip-components=1"
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

        if result.returncode != 0:
            print("ERROR: Download failed")
            print(f"  stderr: {result.stderr}")
            return False

        print("Download complete!")
        return True

    except Exception as e:
        print(f"ERROR: {e}")
        return False


# Get configuration
if IN_DATABRICKS:
    version = dbutils.widgets.get("1_version")
    target_path = dbutils.widgets.get("2_target_path")
else:
    version = DEFAULT_VERSION
    target_path = os.path.join(os.getcwd(), "ai-data-analyst-workshop")

# Execute download
success = download_workshop_materials(version, target_path)

if success:
    print("")
    print(f"Workshop materials downloaded to: {target_path}")
else:
    print("")
    print("Download failed. Please check:")
    print(f"  1. Version exists: https://github.com/{REPO}/releases")
    print("  2. You have internet access")
    print("  3. Target path is writable")

In [ ]:
# Verify Download - List Contents
import os


def verify_download(target_path: str) -> bool:
    """Verify workshop materials were downloaded correctly.

    Args:
        target_path: Path where materials should be extracted

    Returns:
        True if verification passes, False otherwise
    """
    expected_items = [
        "src",
        "notebooks",
        "scripts",
        "config",
        "docs",
        "pyproject.toml",
        "README.md",
    ]

    print(f"Verifying contents of: {target_path}")
    print("=" * 60)

    if not os.path.exists(target_path):
        print("ERROR: Target path does not exist")
        return False

    # List actual contents
    actual_items = os.listdir(target_path)

    print("\nContents:")
    for item in sorted(actual_items):
        item_path = os.path.join(target_path, item)
        if os.path.isdir(item_path):
            sub_items = len(os.listdir(item_path))
            print(f"  {item}/ ({sub_items} items)")
        else:
            size = os.path.getsize(item_path)
            print(f"  {item} ({size:,} bytes)")

    # Check expected items
    print("\nVerification:")
    missing = []
    for item in expected_items:
        if item in actual_items:
            print(f"  [OK] {item}")
        else:
            print(f"  [MISSING] {item}")
            missing.append(item)

    if missing:
        print(f"\nWARNING: {len(missing)} expected items missing")
        return False
    else:
        print("\nAll expected items present!")
        return True


# Run verification
if IN_DATABRICKS:
    target_path = dbutils.widgets.get("2_target_path")
else:
    target_path = os.path.join(os.getcwd(), "ai-data-analyst-workshop")

verified = verify_download(target_path)

if verified:
    print("\n" + "=" * 60)
    print("SUCCESS: Workshop materials ready!")
    print("=" * 60)

## Next Steps

Now that the workshop materials are downloaded:

1. **Set up data**: Open `notebooks/00_setup_workshop_data.ipynb` to load datasets into Unity Catalog
2. **Run the demo**: Open `notebooks/demo.ipynb` for the main AI Data Analyst demonstration
3. **Build your agent**: Try `notebooks/03_build_your_agent.ipynb` for hands-on challenges

### Troubleshooting

**Release not found (404)**:
- Check available releases: https://github.com/compassanalytics/ai-data-analyst-demo/releases
- Verify the version format (e.g., `v1.0` not `1.0`)

**Permission denied**:
- Ensure you have write access to the target directory
- On Databricks, try `/Workspace/Users/your-email@domain.com/workshop`

**curl not found**:
- On Databricks, try `%sh apt-get install curl`
- Or use the Python requests library approach in `docs/workshop_setup.md`